### This example demonstrates the end-to-end workflow for computing a correlation matrix on a very small synthetic dataset generated by a prompt-AI algorithm.

In [2]:
import time
import matplotlib as plt
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

#### Single Thread v Multiprocessing

In [3]:
# Task 1: Single-threaded CPU Baseline
def compute_correlation_serial(X):
    N, T = X.shape
    means = X.mean(axis=1, keepdims=True)
    stds = X.std(axis=1, keepdims=True)
    Z = (X - means) / stds
    C = (Z @ Z.T) / (T - 1)
    return C

# Task 2: Multi-core CPU Parallel Version (Threaded)
def parallel_cpu_correlation_v2(X, num_workers=4):
    """
    Parallel Engine Implementation: MapReduce Pattern
    ------------------------------------------------
    This function implements a CPU-based parallel correlation engine using 
    a 'Split-Apply-Combine' strategy.

    1. Decomposition (Split): 
       Uses np.array_split to divide the standardized dataset Z into 'num_workers' 
       horizontal chunks. This allows each worker to own a subset of the rows.

    2. Orchestration (Map): 
       Utilizes ThreadPoolExecutor to dispatch each chunk to a separate CPU thread. 
       The 'compute_chunk' function performs the matrix multiplication (chunk @ Z.T) 
       independently for its assigned rows.

    3. Assembly (Combine): 
       The 'list(executor.map(...))' serves as a synchronization barrier, waiting 
       for all threads to finish. Finally, np.vstack 'gathers' the partial results 
       to reconstruct the full N x N correlation matrix.

    Note on Performance:
    While logically parallel, this approach may face overhead from thread 
    management and Python's Global Interpreter Lock (GIL), which can result 
    in a speedup factor < 1.0x for smaller N.
    """
    N, T = X.shape
    # Standardizing (Serial part)
    means = X.mean(axis=1, keepdims=True)
    stds = X.std(axis=1, keepdims=True)
    Z = (X - means) / stds

    # Define the worker operation
    def compute_chunk(chunk):
        return (chunk @ Z.T) / (T - 1)

    # Split dataset into N chunks based on number of workers
    chunks = np.array_split(Z, num_workers)
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(compute_chunk, chunks))
        
    return np.vstack(results)

if __name__ == "__main__":
    # Increasing N and T to see meaningful parallel benefits
    # The project requires scaling from small to very large values [cite: 10]
    N, T = 4000, 4000
    np.random.seed(0)
    X = np.random.randn(N, T)

    print(f"Dataset Size: {N} series x {T} time steps")
    print("-" * 40)

    # 1. Run Serial
    t0 = time.time()
    C_serial = compute_correlation_serial(X)
    t1 = time.time()
    serial_time = t1 - t0
    print(f"Serial CPU Time:   {serial_time:.4f} seconds")

    # 2. Run Parallel
    t2 = time.time()
    C_parallel = parallel_cpu_correlation_v2(X, num_workers=4)
    t3 = time.time()
    parallel_time = t3 - t2
    print(f"Parallel CPU Time: {parallel_time:.4f} seconds")

    # 3. Validation 
    correct = np.allclose(C_serial, C_parallel)
    print(f"Correctness Check: {'PASSED' if correct else 'FAILED'}")
    
    # 4. Speedup Analysis 
    print(f"Speedup Factor:    {serial_time / parallel_time:.2f}x")

Dataset Size: 4000 series x 4000 time steps
----------------------------------------
Serial CPU Time:   0.5840 seconds
Parallel CPU Time: 0.7825 seconds
Correctness Check: PASSED
Speedup Factor:    0.75x


# Correlation Matrix Computation: Serial vs Multi-Core CPU Implementation

This project demonstrates an end-to-end workflow for computing a **correlation matrix** from a synthetic dataset and compares the performance of a **single-threaded NumPy implementation** with a **manually parallelized multi-core CPU implementation** in Python.

The goal is to explore how **manual parallelism behaves for numerical workloads** and to understand the **performance trade-offs between Python-level threading and optimized numerical libraries**.

---

## Project Goal

- Implement correlation matrix computation using NumPy
- Compare **serial** and **multi-threaded CPU implementations**
- Analyze when **manual parallelization improves or worsens performance**
- Understand the interaction between **Python threading and BLAS libraries**

---

## Problem Description

Consider a dataset consisting of multiple time series.

- **N** → number of time series  
- **T** → number of time steps  

The dataset can be represented as a matrix:

```
X ∈ ℝ^(N × T)
```

where:

- Each **row** represents a time series
- Each **column** represents a time step

To compute the correlation matrix efficiently, the dataset is first **standardized**, followed by a **matrix multiplication** operation.

---

## Standardization Step

Each time series is normalized so that it has:

- **Zero mean**
- **Unit variance**

This produces the standardized matrix:

```
Z = (X − μ) / σ
```

where:

- **μ** → mean of each time series  
- **σ** → standard deviation of each time series  

Standardization ensures that the correlation calculation is independent of the absolute scale of the data.

---

## Correlation Matrix Computation

After standardization, the correlation matrix is computed using matrix multiplication:

```
C = (Z · Zᵀ) / (T − 1)
```

This produces the matrix:

```
C ∈ ℝ^(N × N)
```

where each element represents the **Pearson correlation coefficient between two time series**.

---

## Implementations Compared

Two implementations are explored in this project.

---

### 1. Serial CPU Baseline

The baseline implementation performs the computation using **NumPy vectorized operations**.

Steps involved:

- Compute mean and standard deviation
- Standardize the dataset
- Perform matrix multiplication `Z @ Z.T`
- Normalize by `(T − 1)`

Although the Python code appears **serial**, the heavy numerical operation is executed using highly optimized **BLAS libraries**, such as:

- **OpenBLAS**
- **Intel MKL**

These libraries are implemented in **low-level languages (C/Fortran)** and automatically use:

- SIMD vectorization
- cache-optimized algorithms
- multi-threaded CPU execution

As a result, the serial implementation is already **highly optimized and capable of using multiple CPU cores internally**.

---

### 2. Multi-Core CPU Parallel Implementation

The second implementation introduces **explicit Python-level parallelism** using:

```
concurrent.futures.ThreadPoolExecutor
```

The approach follows a **Split → Apply → Combine** workflow.

---

#### Split

The standardized matrix `Z` is divided into several horizontal chunks:

```
Z =
[ Z₁
  Z₂
  ...
  Zk ]
```

Each chunk contains a subset of rows assigned to a worker thread.

---

#### Apply

Each worker computes a partial multiplication:

```
chunk · Zᵀ
```

This allows each thread to compute a portion of the correlation matrix independently.

---

#### Combine

After all threads complete their work:

- partial results are collected
- matrices are stacked using `np.vstack`

This reconstructs the final **N × N correlation matrix**.

---

## Performance Observation

Experimental results show that the threaded version may actually run **slower** than the serial implementation.

Example experiment:

```
Dataset Size: 4000 × 4000

Serial CPU Time:   0.58 seconds
Parallel CPU Time: 0.78 seconds
Speedup Factor:    0.75×
```

Instead of accelerating the computation, the parallel implementation introduces **additional overhead**.

---

## Why the Parallel Version is Slower

Several factors explain this behavior.

---

### NumPy Already Uses Parallel BLAS

The operation

```
Z @ Z.T
```

is executed by highly optimized numerical libraries.

These libraries already:

- distribute work across multiple CPU cores
- optimize memory access patterns
- use vectorized CPU instructions

Therefore, the **serial implementation is already parallel at the native level**.

---

### Thread Management Overhead

Introducing Python threads adds extra work:

- thread creation
- task scheduling
- synchronization barriers
- result aggregation

For moderately sized matrices, this overhead can outweigh the benefits of manual parallelism.

---

### Thread Contention

Multiple threads may compete for the same hardware resources:

- CPU cores
- cache memory
- memory bandwidth

Additionally, BLAS libraries may already spawn their own worker threads, leading to **thread oversubscription**.

---

### Memory Bandwidth Bottleneck

Each worker thread repeatedly reads the full matrix `Zᵀ`.

When multiple threads access the same large matrix simultaneously:

- cache efficiency decreases
- memory bandwidth becomes the limiting factor

This further reduces potential speed gains.

---

## Key Insight

This experiment demonstrates an important principle in **scientific computing and high-performance computing (HPC)**:

> Highly optimized numerical libraries often outperform manually parallelized Python code.

For workloads dominated by **linear algebra operations**, relying on optimized libraries like **NumPy + BLAS** is usually the most efficient approach.

---

## Takeaway

Manual parallelization is most beneficial when:

- algorithms involve **Python loops**
- operations **cannot be vectorized**
- tasks are **independent and compute-heavy**

Examples include:

- Monte Carlo simulations
- custom pairwise metrics
- large independent simulations
- non-vectorizable algorithms

For dense linear algebra operations such as matrix multiplication, optimized libraries already provide **near-optimal CPU performance**.
